In [1]:
"""Phase I: Quantil-Prognosen (Median, 75 %, 90 %) für den Nachbestell-Bedarf. Bericht 3.10.
Ausführen: uv run python -u src/quantile.py   (dauert ca. 10 Minuten; Ausgabe: reports/quantile_output.txt, reports/results_quantile.csv, reports/forecast_next16.csv)
Vorher: src/clean.py muss gelaufen sein.
"""

'Phase I: Quantil-Prognosen (Median, 75 %, 90 %) für den Nachbestell-Bedarf. Bericht 3.10.\nAusführen: uv run python -u src/quantile.py   (dauert ca. 10 Minuten; Ausgabe: reports/quantile_output.txt, reports/results_quantile.csv, reports/forecast_next16.csv)\nVorher: src/clean.py muss gelaufen sein.\n'

In [2]:
import sys
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
sys.path.insert(0, str(ROOT / "src"))
from forecast_eval import HORIZON, WINDOWS, load_clean, window_frames  # noqa: E402
from gbm_model import CATEGORICAL, DEFAULT_DROP, PARAMS, N_TREES, Panel  # noqa: E402

pd.set_option("display.width", 170)
lines = []


def out(msg=""):
    print(msg, flush=True)
    lines.append(str(msg))


ALPHAS = (0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9)

## 1. Was ist ein Quantil-Modell?
Das Basismodell (Bericht 3.8) minimiert den quadrierten Fehler auf `log1p(Absatz)` und liefert einen **typischen** Wert.
Hier ändert sich nur die Verlustfunktion: `objective="quantile"` mit `alpha`.
Der Absatz liegt dann in `alpha` aller Fälle **höchstens** so hoch wie die Prognose.
`alpha = 0.9` heißt: an 9 von 10 Tagen reicht der Bestand, an 1 von 10 Tagen nicht.
Quantile bleiben bei monotonen Umrechnungen erhalten, daher ist `expm1` des Log-Quantils das Quantil des Absatzes.
Je Quantil wird ein eigenes Modell trainiert. Danach werden die Werte je Zelle sortiert, damit sich Quantile nicht kreuzen.

In [3]:
def fit_predict_quantiles(panel, origin, alphas=ALPHAS, n_days=730, draws=2, n_trees=N_TREES, drop=DEFAULT_DROP, seed=0):
    """Je alpha ein LightGBM-Modell; Rückgabe: {alpha: Tabelle Zieldatum x Reihe} in Stück (nicht log)."""
    o = panel.dates.get_loc(origin)
    X, y = panel.training_set(o, n_days=n_days, draws=draws, seed=seed)
    X = X.drop(columns=list(drop))
    cats = [c for c in CATEGORICAL if c not in drop]
    Xf, j = panel.forecast_set(o)
    raw = {}
    for a in alphas:
        params = {**PARAMS, "objective": "quantile", "alpha": a, "seed": 42 + seed}
        model = lgb.train(params, lgb.Dataset(X, label=y, categorical_feature=cats, free_raw_data=False), num_boost_round=n_trees)
        raw[a] = model.predict(Xf[X.columns])
    stacked = np.sort(np.stack([raw[a] for a in alphas]), axis=0)  # Quantile nicht kreuzen lassen
    return {
        a: pd.DataFrame(np.clip(np.expm1(stacked[k]), 0, None).reshape(len(j), panel.S), index=panel.dates[j], columns=panel.cols)
        for k, a in enumerate(alphas)
    }


def pinball_log(truth, pred, a):
    """Mittlerer Pinball-Verlust auf log1p-Skala (das Trainingsziel eines Quantil-Modells)."""
    d = np.log1p(truth) - np.log1p(pred)
    return float(np.nanmean(np.maximum(a * d, (a - 1) * d).to_numpy(dtype=float)))

## 2. Prüfung in den drei Fenstern
**Abdeckung** = Anteil der Zellen, in denen der echte Absatz <= Prognose war. Soll ≈ alpha sein.
**Aufschlag** = Mittel der Quantil-Prognose in Stück geteilt durch die des Medians (wie viel mehr Bestand das kostet).
**Summe 16 Tage**: Die Tagesprognosen werden je Reihe aufsummiert und mit dem echten 16-Tage-Absatz verglichen.
Dabei werden nur Reihen ohne Schließtage im Fenster gezählt.

In [4]:
df = load_clean()
panel = Panel(df)
rows, fam_cov = [], {}
for name, origin in WINDOWS.items():
    t0 = time.time()
    preds = fit_predict_quantiles(panel, origin)
    truth, excluded, _ = window_frames(df, origin)
    valid = truth.notna() & ~excluded
    t_eff = truth.where(valid)
    full_series = valid.all(axis=0)  # Reihen mit 16 gültigen Tagen
    total_truth = truth.loc[:, full_series].sum()
    for a, p in preds.items():
        hit = (t_eff <= p).where(valid)
        cov = float(np.nanmean(hit.to_numpy(dtype=float)))
        cov_total = float((total_truth <= p.loc[:, full_series].sum()).mean())
        rows.append({
            "Fenster": name, "alpha": a, "Abdeckung": cov, "Abdeckung Summe 16 Tage": cov_total,
            "Pinball (log)": pinball_log(t_eff, p, a),
            "Aufschlag vs. Median": float(p.where(valid).sum().sum() / preds[0.5].where(valid).sum().sum()),
        })
        if a == 0.9 and name.startswith("W1"):
            fam = pd.Series(hit.columns.get_level_values("family"), index=hit.columns)
            fam_cov = {f: float(np.nanmean(hit[list(c.index)].to_numpy(dtype=float))) for f, c in fam.groupby(fam)}
    out(f"{name}: fertig ({time.time() - t0:.0f} s)")

res = pd.DataFrame(rows)
out("\n" + res.round(4).to_string(index=False))
avg = res.groupby("alpha")[["Abdeckung", "Abdeckung Summe 16 Tage", "Pinball (log)", "Aufschlag vs. Median"]].mean()
out("\nMittel der drei Fenster:\n" + avg.round(4).to_string())

W1 Aug 2017: fertig (249 s)


W2 Dez 2016: fertig (224 s)


W3 Mär 2017: fertig (246 s)



    Fenster  alpha  Abdeckung  Abdeckung Summe 16 Tage  Pinball (log)  Aufschlag vs. Median
W1 Aug 2017   0.50     0.6098                   0.5320         0.1316                1.0000
W1 Aug 2017   0.60     0.6903                   0.7127         0.1284                1.0373
W1 Aug 2017   0.70     0.7656                   0.8507         0.1161                1.0763
W1 Aug 2017   0.75     0.8049                   0.9063         0.1066                1.1015
W1 Aug 2017   0.80     0.8424                   0.9489         0.0954                1.1292
W1 Aug 2017   0.85     0.8819                   0.9731         0.0807                1.1621
W1 Aug 2017   0.90     0.9195                   0.9854         0.0618                1.2069
W2 Dez 2016   0.50     0.5552                   0.4162         0.1579                1.0000
W2 Dez 2016   0.60     0.6429                   0.5941         0.1511                1.0624
W2 Dez 2016   0.70     0.7179                   0.7404         0.1384          


Mittel der drei Fenster:
       Abdeckung  Abdeckung Summe 16 Tage  Pinball (log)  Aufschlag vs. Median
alpha                                                                         
0.50      0.5632                   0.4318         0.1414                1.0000
0.60      0.6475                   0.6197         0.1368                1.0451
0.70      0.7255                   0.7801         0.1240                1.0869
0.75      0.7732                   0.8575         0.1131                1.1223
0.80      0.8164                   0.9107         0.1001                1.1549
0.85      0.8603                   0.9439         0.0844                1.1947
0.90      0.9045                   0.9690         0.0643                1.2499


## 2b. Welches Quantil passt für die 16-Tage-Summe?
Wer für 16 Tage auf einmal bestellt, braucht nicht das 90-%-Quantil jedes Tages, sondern ein Quantil, dessen **Summe** in 90 % der Fälle reicht.
Gesucht: das alpha, bei dem „Abdeckung Summe 16 Tage“ nahe 0,9 liegt. Es wurde auf denselben drei Fenstern gesucht, an denen es geprüft wird;
daher zählt vor allem, ob die Fenster einzeln nahe beieinander liegen.

In [5]:
tot = res.pivot(index="alpha", columns="Fenster", values="Abdeckung Summe 16 Tage")
tot["Mittel"] = tot.mean(axis=1)
tot["Aufschlag vs. Median"] = res.groupby("alpha")["Aufschlag vs. Median"].mean()
out("\nAbdeckung der 16-Tage-Summe je alpha:\n" + tot.round(3).to_string())
best = (tot.Mittel - 0.9).abs().idxmin()
out(f"\nalpha mit Summen-Abdeckung am nächsten an 90 %: {best}")


Abdeckung der 16-Tage-Summe je alpha:
Fenster  W1 Aug 2017  W2 Dez 2016  W3 Mär 2017  Mittel  Aufschlag vs. Median
alpha                                                                       
0.50           0.532        0.416        0.347   0.432                 1.000
0.60           0.713        0.594        0.552   0.620                 1.045
0.70           0.851        0.740        0.749   0.780                 1.087
0.75           0.906        0.813        0.853   0.857                 1.122
0.80           0.949        0.875        0.909   0.911                 1.155
0.85           0.973        0.914        0.945   0.944                 1.195
0.90           0.985        0.950        0.971   0.969                 1.250



alpha mit Summen-Abdeckung am nächsten an 90 %: 0.8


## 3. Wo ist das 90-%-Quantil zu knapp? (Fenster W1)
Familien mit der niedrigsten Abdeckung. Dort reicht der Bestand häufiger nicht, als das Ziel von 90 % verspricht.

In [6]:
fc = pd.Series(fam_cov).sort_values()
out("\nAbdeckung des 90-%-Quantils je Familie (W1), niedrigste 5:\n" + fc.head(5).map("{:.1%}".format).to_string())
out("\nhöchste 5:\n" + fc.tail(5).map("{:.1%}".format).to_string())


Abdeckung des 90-%-Quantils je Familie (W1), niedrigste 5:
SCHOOL AND OFFICE SUPPLIES    78.8%
BEAUTY                        86.0%
MAGAZINES                     86.7%
LINGERIE                      87.5%
HOME AND KITCHEN II           88.8%



höchste 5:
LADIESWEAR    95.9%
PRODUCE       96.1%
DAIRY         96.3%
HOME CARE     96.5%
BOOKS         99.2%


## 4. Prognose für die nächsten 16 Tage (16.–31.08.2017)
Ursprung = letzter Trainingstag 2017-08-15. Kalender, Feiertage und Aktionen der Tage danach stehen in `test.csv`.
Echte Werte gibt es nicht, die Prognose ist eine Demonstration. Vertrauen kommt aus den Abdeckungen oben.

In [7]:
test = pd.read_parquet(ROOT / "data/clean/test_clean.parquet")
test["family"] = test.family.astype(str)
test["is_outlier"] = 0
ext = pd.concat([df, test], ignore_index=True)
origin = df.date.max()
preds = fit_predict_quantiles(Panel(ext), origin)
long = []
for a, p in preds.items():
    s = p.stack(["store_nbr", "family"]).rename(f"q{round(a * 100)}")
    long.append(s)
fc_out = pd.concat(long, axis=1).round(2).reset_index().rename(columns={"level_0": "date"})
fc_out = fc_out.rename(columns={fc_out.columns[0]: "date"})
closed = test.set_index(["date", "store_nbr", "family"])[["is_closed_day", "is_store_closed"]].max(axis=1).rename("geschlossen")
fc_out = fc_out.merge(closed.reset_index(), on=["date", "store_nbr", "family"], how="left")
fc_out.to_csv(ROOT / "reports" / "forecast_next16.csv", index=False)
out(f"\nPrognose geschrieben: {len(fc_out)} Zeilen ({fc_out.date.min().date()} bis {fc_out.date.max().date()})")
tot = fc_out[fc_out.geschlossen == 0].groupby("family")[["q50", "q75", "q90"]].sum().sort_values("q50", ascending=False)
out("\nSumme 16 Tage je Familie (Stück), größte 5, nur offene Tage, Summe der Tageswerte:\n" + tot.head(5).round(0).to_string())


Prognose geschrieben: 28512 Zeilen (2017-08-16 bis 2017-08-31)



Summe 16 Tage je Familie (Stück), größte 5, nur offene Tage, Summe der Tageswerte:
                 q50        q75        q90
family                                    
GROCERY I  3547768.0  3794658.0  4075144.0
BEVERAGES  2719054.0  2960340.0  3269086.0
PRODUCE    1906053.0  2032078.0  2175167.0
CLEANING    889508.0   982183.0  1101376.0
DAIRY       650871.0   702755.0   754610.0


In [8]:
res.to_csv(ROOT / "reports" / "results_quantile.csv", index=False)
(ROOT / "reports" / "quantile_output.txt").write_text("\n".join(lines), encoding="utf-8")

4431